# Workflow – Installation et exécution sur Google Colab

Ce notebook peut être exécuté directement sur Google Colab sans monter Google Drive. Les étapes suivantes sont insérées automatiquement au début du notebook.


In [ ]:
# Cloner le dépôt et installer les dépendances
!git clone https://github.com/Houssam365/Classification-de-S-ries-Temporelles-M-dicales.git
%cd Classification-de-S-ries-Temporelles-M-dicales
!pip install -q -r requirements.txt
# Télécharger les données MIT‑BIH (script du projet)
!python scripts/download_data.py


# Classification de Séries Temporelles Médicales — ECG MIT-BIH

Ce notebook constitue le **point d'entrée pédagogique** du projet.  
Il suit un fil conducteur progressif :

| Étape | Contenu |
|-------|---------|
| 1 | Mise en place de l'environnement |
| 2 | Chargement et exploration des données ECG |
| 3 | Analyse exploratoire : visualisation des classes |
| 4 | Baseline : features manuelles + Random Forest |
| 5 | Modèles deep learning : CNN1D, ResNet1D, CNN-LSTM |
| 6 | Évaluation clinique : métriques médicales |
| 7 | Interprétabilité : GradCAM, attention, filtres |
| 8 | Comparaison des architectures |

> **Rappel** : la stratification train/val/test est réalisée **par patient** (enregistrement)
> pour éviter toute fuite de données (data leakage).


---
## Étape 1 — Mise en place de l'environnement


In [ ]:
# Sur Google Colab : cloner le dépôt et installer les dépendances
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone https://github.com/Houssam365/Classification-de-S-ries-Temporelles-M-dicales.git
    %cd Classification-de-S-ries-Temporelles-M-dicales
    %pip install -q -r requirements.txt

# En local : s'assurer que la racine du projet est dans le PYTHONPATH
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from pathlib import Path

# Modules du projet
from scripts.download_data import download_mitdb
from src.data_processing   import ECGDataset, CLASS_NAMES, WINDOW_SIZE
from src.augmentation      import jitter, time_warp, magnitude_warp, oversample_minority_classes
from src.models            import CNN1D, ResNet1D, CNN_LSTM, HandcraftedFeatureExtractor, build_model
from src.train             import Trainer, make_loaders, set_seed
from src.evaluate          import predict, compute_metrics, plot_confusion_matrix, plot_metrics_by_class, plot_learning_curves, compare_models
from src.interpretability  import GradCAM1D, plot_gradcam, plot_attention_weights, plot_conv_filters, plot_misclassified

# Paramètres globaux
SEED    = 42
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
FS      = 360.0  # Hz

set_seed(SEED)
print(f'Device : {DEVICE}')
print(f'PyTorch : {torch.__version__}')

---
## Étape 2 — Téléchargement et chargement des données MIT-BIH

La base MIT-BIH Arrhythmia contient **48 enregistrements ECG** de 30 minutes chacun,
annotés battement par battement par des cardiologues.
La fréquence d'échantillonnage est de **360 Hz** (360 points par seconde).

Le module `data_processing` segmente automatiquement ces signaux continus en fenêtres
de **360 échantillons** (1 seconde) centrées sur chaque pic R annoté.


In [ ]:
# Téléchargement (ignoré si les fichiers existent déjà)
data_dir = download_mitdb()
decoded_dir = data_dir / 'decoded'
print(f'Données dans : {decoded_dir}')
print(f'Enregistrements disponibles : {len(list(decoded_dir.glob("*.npz")))}')

In [ ]:
# Chargement et segmentation de tous les enregistrements
dataset = ECGDataset(decoded_dir, seed=SEED).load()

# Découpage train / val / test STRATIFIÉ PAR PATIENT
(X_train, y_train), (X_val, y_val), (X_test, y_test) = dataset.split()

print(f'Train : {len(y_train):>7,} battements')
print(f'Val   : {len(y_val):>7,} battements')
print(f'Test  : {len(y_test):>7,} battements')
print(f'Taille d\'une fenêtre : {WINDOW_SIZE} échantillons ({WINDOW_SIZE/FS:.2f} s)')

---
## Étape 3 — Analyse exploratoire des données

Avant toute modélisation, il est indispensable de **comprendre les données** :
- Distribution des classes (déséquilibre ?)
- Morphologie typique de chaque classe
- Variabilité intra-classe


In [ ]:
# Tableau descriptif du déséquilibre de classes
df_desc = dataset.describe()
display(df_desc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(df_desc['Classe'], df_desc['Nombre de beats'], color='steelblue')
axes[0].set_title('Distribution des classes (échelle linéaire)')
axes[0].set_ylabel('Nombre de battements')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(df_desc['Classe'], df_desc['Nombre de beats'], color='tomato', log=True)
axes[1].set_title('Distribution des classes (échelle logarithmique)')
axes[1].set_ylabel('Nombre de battements (log)')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Déséquilibre de classes MIT-BIH', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nObservation : la classe N (Normal) représente ~80 % des battements.')
print('→ L\'accuracy seule est une métrique trompeuse.')
print('→ On utilisera le Macro F1 et la Sensibilité par classe.')

In [ ]:
# Affichage d'exemples de signaux ECG par classe
t = np.arange(WINDOW_SIZE) / FS  # axe temporel en secondes

fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(16, 3 * len(CLASS_NAMES)))

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    idx_cls = np.where(y_train == cls_idx)[0]
    if len(idx_cls) == 0:
        continue
    chosen  = idx_cls[:4]
    for col, idx in enumerate(chosen):
        axes[cls_idx, col].plot(t, X_train[idx], lw=1.0, color='navy')
        if col == 0:
            axes[cls_idx, col].set_ylabel(cls_name, fontsize=10, fontweight='bold')
        axes[cls_idx, col].set_xlabel('Temps (s)', fontsize=8)
        axes[cls_idx, col].grid(alpha=0.3)
        axes[cls_idx, col].set_title(f'Exemple {col + 1}', fontsize=8)

plt.suptitle('Exemples de battements ECG par classe AAMI', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Démonstration des techniques d'augmentation
example = X_train[y_train == 2][0:1]  # Un battement ventriculaire (V)

augmented = {
    'Original':         example[0],
    'Jittering':        jitter(example)[0],
    'Time Warping':     time_warp(example)[0],
    'Magnitude Warping': magnitude_warp(example)[0],
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, (name, sig) in zip(axes, augmented.items()):
    ax.plot(t, sig, lw=1.0, color='darkgreen' if name == 'Original' else 'steelblue')
    ax.set_title(name, fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_xlabel('Temps (s)', fontsize=8)

plt.suptitle('Techniques d\'augmentation — Battement ventriculaire (V)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Étape 4 — Baseline : features manuelles + Random Forest

Avant les modèles deep learning, on établit une **baseline solide** avec :
- Extraction de features statistiques et fréquentielles sur chaque battement
- Classifieur Random Forest

Cette baseline sert de **seuil minimal** à dépasser et permet de quantifier
l'apport réel du deep learning.


In [ ]:
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler

print('Extraction des features manuelles…')
extractor = HandcraftedFeatureExtractor()
X_train_feat = extractor.transform(X_train)
X_val_feat   = extractor.transform(X_val)
X_test_feat  = extractor.transform(X_test)

print(f'Dimension des features : {X_train_feat.shape[1]} par battement')

# Normalisation (importante pour certains classifieurs, bonne pratique générale)
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_val_feat   = scaler.transform(X_val_feat)
X_test_feat  = scaler.transform(X_test_feat)

print('Entraînement du Random Forest…')
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED, class_weight='balanced')
rf.fit(X_train_feat, y_train)

y_pred_rf = rf.predict(X_test_feat)
metrics_rf = compute_metrics(y_test, y_pred_rf, class_names=CLASS_NAMES)

print('\n── Rapport de classification (Random Forest) ──')
print(metrics_rf['report_str'])
print(f'Macro F1  : {metrics_rf["macro_f1"]:.4f}')
print(f'Accuracy  : {metrics_rf["accuracy"]:.4f}')

plot_confusion_matrix(metrics_rf['conf_matrix'], CLASS_NAMES, title='Random Forest')

---
## Étape 5 — Modèles Deep Learning

Trois architectures sont comparées :

| Architecture | Philosophie | Points forts |
|---|---|---|
| **CNN1D** | Motifs locaux par convolution | Léger, rapide à entraîner |
| **ResNet1D** | Connexions résiduelles → réseau profond | Meilleure généralisation |
| **CNN-LSTM** | Local (CNN) + temporel long (LSTM) + attention | Interprétabilité par attention |

> **Rééquilibrage** : avant l'entraînement, les classes minoritaires sont
> sur-échantillonnées par augmentation pour atteindre au minimum 30 % de la classe majoritaire.


In [ ]:
# Rééquilibrage par sur-échantillonnage augmenté
print(f'Avant rééquilibrage : {len(y_train):,} battements')
X_train_bal, y_train_bal = oversample_minority_classes(X_train, y_train, target_ratio=0.3, seed=SEED)
print(f'Après rééquilibrage : {len(y_train_bal):,} battements')

# Vérification de la nouvelle distribution
classes, counts = np.unique(y_train_bal, return_counts=True)
for cls, cnt in zip(classes, counts):
    print(f'  {CLASS_NAMES[cls]}: {cnt:,}')

In [ ]:
BATCH_SIZE = 256
train_loader, val_loader = make_loaders(X_train_bal, y_train_bal, X_val, y_val, BATCH_SIZE)

In [ ]:
# ── 5a. CNN1D ──
set_seed(SEED)
model_cnn = CNN1D(n_classes=5)
trainer_cnn = Trainer(model_cnn, epochs=30, lr=1e-3, save_dir='../results')
history_cnn = trainer_cnn.fit(train_loader, val_loader, y_train_bal, model_name='cnn1d')
plot_learning_curves(history_cnn, model_name='CNN1D')

In [ ]:
# ── 5b. ResNet1D ──
set_seed(SEED)
model_resnet = ResNet1D(n_classes=5)
trainer_resnet = Trainer(model_resnet, epochs=30, lr=1e-3, save_dir='../results')
history_resnet = trainer_resnet.fit(train_loader, val_loader, y_train_bal, model_name='resnet1d')
plot_learning_curves(history_resnet, model_name='ResNet1D')

In [ ]:
# ── 5c. CNN-LSTM avec attention ──
set_seed(SEED)
model_cnnlstm = CNN_LSTM(n_classes=5)
trainer_cnnlstm = Trainer(model_cnnlstm, epochs=30, lr=5e-4, save_dir='../results')
history_cnnlstm = trainer_cnnlstm.fit(train_loader, val_loader, y_train_bal, model_name='cnn_lstm')
plot_learning_curves(history_cnnlstm, model_name='CNN-LSTM')

---
## Étape 6 — Évaluation clinique sur le jeu de test

L'évaluation est réalisée sur des **patients jamais vus** pendant l'entraînement.
On calcule les métriques médicales recommandées par la norme **AAMI EC57** :
sensibilité, spécificité, VPP, F1 par classe, Macro F1.


In [ ]:
all_results = {}

for name, model in [('CNN1D', model_cnn), ('ResNet1D', model_resnet), ('CNN-LSTM', model_cnnlstm)]:
    y_pred, y_prob = predict(model, X_test, device=DEVICE)
    metrics = compute_metrics(y_test, y_pred, class_names=CLASS_NAMES)
    all_results[name] = metrics

    print(f'\n══ {name} ══')
    print(metrics['report_str'])
    plot_confusion_matrix(metrics['conf_matrix'], CLASS_NAMES, title=f'Confusion — {name}',
                          save_path=f'../results/{name.lower()}_confusion.png')
    plot_metrics_by_class(metrics, CLASS_NAMES,
                          save_path=f'../results/{name.lower()}_metrics.png')

In [ ]:
# Tableau récapitulatif
compare_models(all_results)

# Graphique de comparaison du Macro F1
names    = list(all_results.keys())
macro_f1 = [all_results[n]['macro_f1'] for n in names]
accuracy = [all_results[n]['accuracy']  for n in names]

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, macro_f1, 0.4, label='Macro F1', color='steelblue')
ax.bar(x + 0.2, accuracy,  0.4, label='Accuracy',  color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Comparaison des architectures sur le jeu de test')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/comparison.png', dpi=150)
plt.show()

---
## Étape 7 — Interprétabilité

L'interprétabilité est fondamentale en médecine : le modèle doit pouvoir
**justifier ses décisions** pour être validé par des cardiologues.

- **GradCAM** : montre *où* le CNN regarde dans le signal
- **Attention** : montre *quels instants* le LSTM considère critiques
- **Filtres** : révèle les *motifs primitifs* appris par le CNN


In [ ]:
# GradCAM sur le ResNet1D
# On cible la dernière couche convolutive du dernier bloc résiduel
target_layer = list(model_resnet.layer3.modules())[-1]  # dernière BN du bloc 3
# Retrouver la dernière Conv1d
for m in model_resnet.layer3.modules():
    if isinstance(m, torch.nn.Conv1d):
        target_layer = m

gradcam = GradCAM1D(model_resnet, target_layer, device=torch.device(DEVICE))

# Exemple : un battement ventriculaire (V)
idx_v   = np.where(y_test == 2)[0][0]
x_v     = torch.from_numpy(X_test[idx_v:idx_v+1]).unsqueeze(1).float()
heatmap = gradcam.compute(x_v, class_idx=2)

plot_gradcam(
    signal   = X_test[idx_v],
    heatmap  = heatmap,
    fs       = FS,
    title    = 'GradCAM — Battement ventriculaire (V) — ResNet1D',
    save_path= '../results/gradcam_ventricular.png',
)
gradcam.remove_hooks()

In [ ]:
# Poids d'attention CNN-LSTM sur un battement supraventriculaire (S)
model_cnnlstm.eval()
idx_s = np.where(y_test == 1)[0][0]
x_s   = torch.from_numpy(X_test[idx_s:idx_s+1]).unsqueeze(1).float().to(DEVICE)

with torch.no_grad():
    _, attn_weights = model_cnnlstm(x_s)

attn_np = attn_weights[0].cpu().numpy()

plot_attention_weights(
    signal       = X_test[idx_s],
    attn_weights = attn_np,
    fs           = FS,
    title        = 'Attention — Battement supraventriculaire (S) — CNN-LSTM',
    save_path    = '../results/attention_supraventricular.png',
)

In [ ]:
# Filtres convolutifs de la première couche CNN1D
plot_conv_filters(model_cnn, n_filters=16, save_path='../results/conv_filters.png')

In [ ]:
# Analyse des exemples mal classés par le meilleur modèle
y_pred_best, _ = predict(model_resnet, X_test, device=DEVICE)
plot_misclassified(
    X_test, y_test, y_pred_best, CLASS_NAMES, fs=FS,
    save_path='../results/misclassified.png'
)

---
## Conclusion

Ce projet a démontré une approche complète de classification de séries temporelles médicales :

### Apports méthodologiques
- **Stratification par patient** : évite le data leakage, donne une estimation réaliste des performances
- **Augmentation ciblée** : rééquilibre les classes sans distordre la physiologie
- **Loss pondérée** : accorde plus d'importance aux arythmies critiques rares
- **Métriques médicales** : sensibilité, spécificité et VPP plutôt que la seule accuracy

### Comparaison des architectures
| Architecture | Avantage | Limite |
|---|---|---|
| Random Forest | Interprétable, rapide | Performances limitées sur les classes rares |
| CNN1D | Léger, bon compromis | Ignore les dépendances longues |
| ResNet1D | Profond, généralise bien | Boîte noire |
| CNN-LSTM | Capture long terme + attention | Plus lent, plus complexe |

### Pistes d'amélioration
- Transfer learning depuis des ECG pré-entraînés sur de plus grandes bases
- Transformer / Self-Attention pour remplacer le LSTM
- Utilisation multi-leads (12 dérivations) pour les bases PTB/PTBXL
